# Ejercicio 2
Comparación de modelos en tres experimentos.

In [1]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent / "src"))

import pandas as pd
from lista4.data import load_csv
from lista4.models import fit_linear, linear_predictions
from lista4.metrics import empirical_risk

archivos = [
    "experimento_2_a.csv",
    "experimento_2_b.csv",
    "experimento_2_c.csv"
]

filas = []

for archivo in archivos:
    df = load_csv(archivo)

    m_x1 = fit_linear(df, ["x1"])
    m_x1x2 = fit_linear(df, ["x1", "x2"])

    p_x1 = linear_predictions(m_x1, df, ["x1"])
    p_x1x2 = linear_predictions(m_x1x2, df, ["x1", "x2"])

    r_x1 = empirical_risk(df["y"], p_x1)
    r_x1x2 = empirical_risk(df["y"], p_x1x2)

    filas.append({
        "archivo": archivo,
        "corr(x1,x2)": df["x1"].corr(df["x2"]),
        "R_x1": r_x1,
        "R_x1x2": r_x1x2,
        "beta_1": m_x1x2.coef_[0],
        "beta_2": m_x1x2.coef_[1],
        "Delta_R": r_x1 - r_x1x2
    })

resultados = pd.DataFrame(filas)
resultados


,archivo,"corr(x1,x2)",R_x1,R_x1x2,beta_1,beta_2,Delta_R
0,experimento_2_a.csv,0.999760,1.029350,1.025907,5.957231,-2.900650,0.003443
1,experimento_2_b.csv,0.999779,1.061539,1.059304,0.606315,2.325776,0.002235
2,experimento_2_c.csv,0.999821,1.141281,1.140041,4.903450,-1.911232,0.001240



Al calcular `corr(x1, x2)` en los tres archivos, nos damos cuenta de que en todos
el valor es prácticamente 1. O sea, `x1` y `x2` están casi perfectamente
correlacionados entre sí. Esto quiere decir que hay muchísima colinealidad: las
dos variables están diciendo casi lo mismo sobre `y`.

En la tabla tenemos el riesgo empírico de los dos modelos en cada archivo, más
los coeficientes `beta1` y `beta2` del modelo conjunto.

Lo que vemos es que el riesgo del modelo con ambas variables es solo un poquito
menor que el del modelo con `x1` solo. La mejora está ahí (tiene que estar,
porque el modelo simple es un caso particular del conjunto), pero es mínima.


La tabla nos resume por archivo: la correlación entre `x1` y `x2`, los dos
riesgos (`R_x1` y `R_x1_x2`), la diferencia `Delta_R` entre ellos, y los
coeficientes `beta1` y `beta2` del modelo con ambas variables.


Para cada archivo calculamos:

`R_x1`, el riesgo del modelo que usa solo `x1`.
`R_x1_x2`, el riesgo del modelo con `x1` y `x2`.
`Delta_R = R_x1 - R_x1_x2`, que es cuánto baja el riesgo cuando agregamos `x2`.

Los tres valores de `Delta_R` nos dan chiquitos. Eso ya nos está diciendo que
agregar `x2` no cambia casi nada: `x1` por sí sola ya estaba explicando `y`.

In [2]:
rangos = pd.Series({
    "rango_beta_1": resultados["beta_1"].max() - resultados["beta_1"].min(),
    "rango_beta_2": resultados["beta_2"].max() - resultados["beta_2"].min(),
    "rango_Delta_R": resultados["Delta_R"].max() - resultados["Delta_R"].min()
})
rangos


rango_beta_1     5.350916
rango_beta_2     5.226426
rango_Delta_R    0.002203
dtype: float64

El rango de `beta1` es de más de 5 unidades.
El rango de `beta2` también es de más de 5 unidades.
Pero el rango de `Delta_R` es de apenas 0.002.

O sea: los coeficientes se mueven un montón de un archivo a otro, pero la
reducción del riesgo al agregar `x2` es prácticamente la misma en los tres. Esa
diferencia tan grande ya nos está adelantando la conclusión del siguiente punto.

### Respuesta (f)

Agregar `x2` casi no cambia el riesgo porque los tres `Delta_R` son diminutos
comparados con los riesgos. Creemos que es porque `x1` y `x2` están casi
perfectamente correlacionados y `x2` no aporta nada nuevo.

Los coeficientes sí cambian muchísimo entre archivos. `beta2` incluso cambia de
signo. Eso pasa por la colinealidad alta: el modelo reparte el peso entre `x1` y
`x2` como quiere sin que el ajuste empeore, entonces los coeficientes individuales
pierden sentido.

En resumen, con colinealidad fuerte el modelo predice bien pero los coeficientes
no se pueden interpretar por separado.